## LIBRARIES

In [ ]:
import optuna
import torch 
import torch.nn as nn
import matplotlib.pyplot as plt 
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random



## DEVICE_TO_GPU

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#seed

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)

## HYPERPARAMS

In [ ]:
EPOCHS = 25
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3



## IMAGE TRANSFORMATIONS


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

## FETCHING DATASET

In [ ]:
train_dataset = datasets.CIFAR10(root ="data", train = True , download = True, transform = transform)

In [ ]:
test_dataset = datasets.CIFAR10(root ="data", train = False , download = True, transform = transform)

## CREATING VISION TRANSFORMER


In [ ]:
class PatchEmbedding(nn.Module):

    def __init__(self, img_Size , patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels=in_channels, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)

        num_patches = (img_Size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1 ,1 ,embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1 ,1+num_patches, embed_dim))

    def forward(self , x: torch.tensor):

        B = x.size(0)
        x = self.proj(x) # ( B, E, H/P, W/P)
        x = x.flatten(2).transpose(1,2) #(B, N, E)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed

        return x


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop_rate):
        super().__init__()
        self.fc1 = nn.Linear(in_features=in_features, out_features=hidden_features)
        self.fc2 = nn.Linear(in_features=hidden_features, out_features=in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.dropout(F.gelu(self.fc1(x)))
        x = self.dropout(self.fc2(x))

        return x

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout = drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))

        return x

In [ ]:
class VisionTransformer(nn.Module):

    def __init__(self, img_size, patch_size, in_channels, num_classes, embed_dim, depth, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        self.encoder = nn.Sequential(*[
            TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)


    def forward(self, x):

        x = self.patch_embed(x)
        x = self.encoder(x)
        x = self.norm(x)
        cls_token = x[:,0]
        return self.head(cls_token)

## DEFINING TRAINING LOOP FUNCTION

In [ ]:
def train(model, loader, optimizer, criterion):

    model.train()

    total_loss, correct = 0, 0 

    for x, y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out , y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1)== y).sum().item()
        # you have to scale the loss(normalization to make the loss general across all the batches)
    return total_loss/ len(loader.dataset), correct/len(loader.dataset)

In [ ]:
def evaluate(model, loader):

    model.eval() # set the mode of the model to evaluation
    correct = 0 
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(dim=1)==y).sum().item()

    return correct/len(loader.dataset)


In [ ]:
from tqdm.auto import tqdm

In [ ]:
def predict_and_plot_grid(model, dataset, classes, grid_size=3):
    model.eval()
    fig,axes = plt.subplots(grid_size, grid_size, figsize=(9,9))
    for i in range(grid_size):
        for j in range(grid_size):
            idx = random.randint(0, len(dataset)-1)
            img, true_label = dataset[idx]
            input_tensor = img.unsqueeze(dim=0).to(device)
            with torch.inference_mode():
                output = model(input_tensor)
                _, predicted = torch.max(output.data, 1) 
            img = img /2 + 0.5 # Unnormalize, to make negative pixel values to positive to be able to plot them with matplotlib
            npimg = img.cpu().numpy()
            axes[i,j].imshow(np.transpose(npimg, (1,2,0)))
            truth = classes[true_label] == classes[predicted.item()]
            if truth:
                color ="g"
            else:
                color = "r"

            axes[i,j].set_title(f"Truth: {classes[true_label]}\n, Predicted: {classes[predicted.item()]}", fontsize=10, c= color)
            axes[i,j].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
def objective(trial):
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    embed_dim = trial.suggest_categorical("embed_dim", [128, 256, 512])
    num_heads = trial.suggest_categorical("num_heads", [4, 8, 16])
    depth = trial.suggest_int("depth", 4, 8)
    mlp_dim = trial.suggest_categorical("mlp_dim", [256, 512, 1024])
    drop_rate = trial.suggest_float("drop_rate", 0.0, 0.5)
    
    # DataLoaders
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)
    
    # Model Setup
    model = VisionTransformer(
        IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES, embed_dim, depth, num_heads, mlp_dim, drop_rate
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
    
    # Training Loop
    for epoch in range(EPOCHS):
        train_loss, train_acc = train(model, train_loader, optimizer, criterion)
        test_acc = evaluate(model, test_loader)
        
        # Report for pruning
        trial.report(test_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return test_acc


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


In [ ]:
best = study.best_params

final_model = VisionTransformer(
    IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES,
    best["embed_dim"], best["depth"], best["num_heads"],
    best["mlp_dim"], best["drop_rate"]
).to(device)

optimizer = torch.optim.Adam(final_model.parameters(), lr=best["learning_rate"])
train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=best["batch_size"],
    shuffle=False
)

train_accuracies, test_accuracies = [], []

for epoch in tqdm(range(EPOCHS)):
    train_loss, train_acc = train(final_model,train_loader, optimizer, nn.CrossEntropyLoss())
    test_acc = evaluate(final_model,test_loader)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    print(f"Epoch:{epoch+1}/{EPOCHS}, train_loss: {train_loss:.4f}, train_accuracy:{train_acc:.4f}, test_accuracy:{test_acc:.4f}")

In [ ]:
predict_and_plot_grid(final_model, test_dataset, classes= train_dataset.classes, grid_size=3)